# LegalDrift Tutorial

Detect semantic drift in legal documents using LegalDrift.

This notebook demonstrates:
1. Basic drift detection between two contracts
2. Document analysis and concept extraction
3. Chunked/localized drift detection
4. Saving and querying drift history

In [ ]:
from legaldrift import (
    LegalDocument,
    EmbeddingEngine,
    DriftDetector,
    LegalConceptExtractor,
    DriftHistory,
    chunk_by_sections,
    align_chunks,
)
import numpy as np

## 1. Basic Drift Detection

Load two versions of a contract and detect if their semantic meaning has drifted.

In [ ]:
contract_v1 = """
This AGREEMENT is entered into as of January 1, 2024.

1. SERVICES
Company shall provide software development services.
Company shall comply with GDPR.

2. PAYMENT
Client shall pay within 30 days of invoice.

3. TERM
This Agreement continues for 1 year.
Either party may terminate with 60 days notice.
"""

contract_v2 = """
This AGREEMENT is entered into as of January 1, 2024.

1. SERVICES
Company shall provide software development and AI training services.
Company shall comply with GDPR and the EU AI Act.

2. PAYMENT
Client shall pay within 15 days of invoice.
All payments must be made in Euros.

3. TERM
This Agreement continues for 2 years.
Either party may terminate with 30 days notice.
"""

doc1 = LegalDocument(text=contract_v1, document_id="contract_v1", jurisdiction="US")
doc2 = LegalDocument(text=contract_v2, document_id="contract_v2", jurisdiction="EU")

print(f"Document 1: {doc1.word_count} words, {doc1.char_count} chars")
print(f"Document 2: {doc2.word_count} words, {doc2.char_count} chars")

In [ ]:
engine = EmbeddingEngine(use_legal_bert=False)
detector = DriftDetector(threshold=0.05)

emb1 = engine.encode([doc1.text])
emb2 = engine.encode([doc2.text])

result = detector.detect(emb1, emb2)

print(f"Drift detected: {result.drift_detected}")
print(f"P-value: {result.p_value:.4f}")
print(f"Confidence: {result.confidence:.2%}")
print(f"Severity: {result.severity:.4f}")
print(f"Effect Size: {result.effect_size:.4f}")
print("\nIndividual Tests:")
for name, data in result.tests.items():
    print(f"  {name}: p={data.get('p_value', 'N/A'):.4f}")

## 2. Concept Extraction

Identify legal concepts like obligations, permissions, and regulatory references.

In [ ]:
extractor = LegalConceptExtractor()

concepts_v1 = extractor.extract_from_text(contract_v1)
concepts_v2 = extractor.extract_from_text(contract_v2)

print("Version 1 concepts:", sorted(concepts_v1))
print("Version 2 concepts:", sorted(concepts_v2))
print("New concepts in v2:", sorted(concepts_v2 - concepts_v1))
print("Removed concepts:", sorted(concepts_v1 - concepts_v2))

## 3. Chunked / Localized Drift Detection

Instead of comparing whole documents, compare section-by-section to find *which part* changed.

In [ ]:
chunks1 = chunk_by_sections(doc1)
chunks2 = chunk_by_sections(doc2)

print(f"Document 1 sections: {len(chunks1)}")
for c in chunks1:
    header = c.metadata.get('header', f'Section {c.chunk_index}')
    print(f"  [{c.chunk_index}] {header[:60]}... ({c.word_count} words)")

print(f"\nDocument 2 sections: {len(chunks2)}")
for c in chunks2:
    header = c.metadata.get('header', f'Section {c.chunk_index}')
    print(f"  [{c.chunk_index}] {header[:60]}... ({c.word_count} words)")

In [ ]:
# Encode and compare aligned sections
aligned = align_chunks(chunks1, chunks2)

print("Section-level drift comparison:")
print("=" * 60)
for c1, c2 in aligned:
    if c1 is None:
        print(f"  [NEW] Section {c2.chunk_index}: added in v2")
        continue
    if c2 is None:
        print(f"  [DEL] Section {c1.chunk_index}: removed in v2")
        continue
    
    e1 = engine.encode([c1.text])
    e2 = engine.encode([c2.text])
    result = detector.detect(e1, e2)
    status = "DRIFT" if result.drift_detected else "OK"
    header = c1.metadata.get('header', f'Section {c1.chunk_index}')[:50]
    print(f"  [{status}] {header}: p={result.p_value:.4f}, severity={result.severity:.4f}")

## 4. Saving and Querying Drift History

Persist drift detection results for trend analysis.

In [ ]:
history = DriftHistory(path="demo_history.json", backend="json")

# Save the full-document drift result
emb1_full = engine.encode([doc1.text])
emb2_full = engine.encode([doc2.text])
result_full = detector.detect(emb1_full, emb2_full)

history.save(
    baseline_id="contract_v1",
    current_id="contract_v2",
    result=result_full,
    notes="Initial comparison after EU AI Act update",
    tags=["gdpr", "ai-act", "contract"],
)

# Query history
records = history.query(limit=10)
print(f"Stored {len(records)} records")
for r in records:
    print(f"  {r.timestamp}: {r.baseline_id} -> {r.current_id} "
          f"(drift={r.result['drift_detected']}, p={r.result['p_value']:.4f})")

In [ ]:
# Cleanup
import os
if os.path.exists("demo_history.json"):
    os.remove("demo_history.json")
print("Demo complete!")